# Replot corrected SVM results

Load one explicit export from `experiment_SVM_new.ipynb`. The raw results contain
termination statuses, the full configuration and environment. This notebook writes
only a new plot alongside the selected export. Choose the run deliberately; no old
reference files or automatic "latest run" selection are used.


In [ ]:
from pathlib import Path
import pickle
import sys
import numpy as np
import matplotlib.pyplot as plt

SVM_DIR = Path.cwd() if (Path.cwd() / "SVM_utils_new.py").exists() else Path.cwd() / "SVM"
sys.path.insert(0, str(SVM_DIR.resolve()))
from SVM_utils_new import plot_figures, plot_for_paper_2


In [ ]:
# Paste the path printed by the corrected experiment notebook.
DATA_FILE = None
if DATA_FILE is None:
    raise ValueError("Set DATA_FILE to a results_new/.../svm_results_full_new.pkl export")
DATA_FILE = Path(DATA_FILE)
with DATA_FILE.open("rb") as result_file:
    results = pickle.load(result_file)
histories_by_label = results["histories"]
labels = results["labels"]
histories = list(histories_by_label.values())
print("Configuration:", results["config"])
print("Termination statuses:", results["statuses"])
print(f"Loaded {len(histories)} histories from {DATA_FILE}")


In [ ]:
colors = ["#E8601C", "pink", "gold", "black", "#9B8AC4", "#88A5DD", "#8DCBE4", "#C2E3D2", "#DDECBF"]
linewidths = [2.0] * len(histories)
linestyles = ["-"] * len(histories)
f_star = results["best_observed_func"]


In [ ]:
for measure, reference, ylabel in [("func", f_star, "Objective gap to best observed"),
                                    ("grad_norm", 0, "Gradient norm")]:
    plot_figures(histories, labels, n=results["config"]["n_features"], m=None, mu_str="0.5",
                 colors=colors, linewidths=linewidths, linestyles=linestyles,
                 f_star=reference, threshold=1e-10, which_measure_1=measure,
                 which_measure_2=measure, ylabel=ylabel)


In [ ]:
import importlib
import SVM_utils_new
importlib.reload(SVM_utils_new)  # Pick up plot-style edits without rerunning the solvers.
from SVM_utils_new import plot_for_paper_2

plot_for_paper_2(
    histories, labels, n=results["config"]["n_features"], m=None, mu_str="0.5",
    colors=colors, linewidths=linewidths, linestyles=linestyles,
    f_star=f_star, threshold=1e-10, max_iter=65, clip_time=None,
    filename=DATA_FILE.with_name("plot_SVM_func_replot_new.pdf"),
)


In [ ]:
for label, history, color in zip(labels, histories, colors):
    times = history["time"]
    iters = range(len(times))
    plt.plot(iters, times, label=label, color=color)

plt.xlabel("Outer iteration / SVRG epoch")
plt.ylabel("Time (seconds)")
plt.legend()
plt.grid(True)
plt.show()